<h2> This code uses Stable baselines3 to train the model instead of mjrl </h2>

Import modules

In [9]:
import myosuite
import gym
import skvideo.io
import numpy as np
import os
from stable_baselines3.common.logger import configure
from IPython.display import HTML
from base64 import b64encode
from sb3_contrib import RecurrentPPO
from stable_baselines3 import PPO
from stable_baselines3 import SAC
import torch
 
def show_video(video_path, video_width = 400):
   
  video_file = open(video_path, "r+b").read()
 
  video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"
  return HTML(f"""<video autoplay width={video_width} controls><source src="{video_url}"></video>""")


tmp_path = "/tmp"
# set up logger
new_logger = configure(tmp_path, ["stdout", "csv", "tensorboard"])

Logging to /tmp


# Vectorize environment

In [10]:
from stable_baselines3.common.vec_env import DummyVecEnv
import gym

env = gym.make('CenterReachOut-v0')

# MLP Training

In [ ]:
# Setup logging directory
logdir = "logs"

model = PPO(
    "MlpPolicy",
    env,
    ent_coef=0.001,
    tensorboard_log=logdir,
    learning_rate=5e-4,
    clip_range=0.1,
    n_steps=512,
    device="cpu",
    policy_kwargs={
        "net_arch": [dict(pi=[128, 128], vf=[128, 128])],
        "activation_fn": torch.nn.Tanh,
    },
    
)
print(f"Using device: {model.device}")

Using device: cpu


# SAC Training

In [28]:
# Setup logging directory
logdir = "logs"

model = SAC(
    "MlpPolicy",
    env,
    ent_coef=0.01,
    tensorboard_log=logdir,
    batch_size=64, 
    device="cpu",
    
)
print(f"Using device: {model.device}")

Using device: cpu


<h3> RNN training </h3>

In [3]:
# Setup logging directory
logdir = "logs"

model = RecurrentPPO(
    "MlpLstmPolicy",
    env,
    tensorboard_log=logdir,
    learning_rate=1e-4,
    clip_range=0.1,
    n_epochs=30,
    batch_size=512,
    n_steps=512,
    device="cpu",
    policy_kwargs=dict(lstm_hidden_size=128, net_arch=[dict(pi=[256], vf=[256])], activation_fn=torch.nn.Tanh),  
)

print(f"Using device: {model.device}")

c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:77: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")
c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\policies.py:486: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  wa

Using device: cpu


# Load Model

In [1]:
model = RecurrentPPO.load("RecurrentPPO", env=env)

NameError: name 'RecurrentPPO' is not defined

# Start Training

In [ ]:
if not os.path.exists(logdir):
    os.makedirs(logdir)

model.learn(total_timesteps=10000000)

In [13]:
model.save('MLP-w-feedback-full-reward') #Saves model 

<h2> Load sb3 model <h2>

In [13]:
model = PPO.load("recurrentNetworkPPO", env=env, device="cuda")

# Render trained policy
frames = []
for _ in range(13): # 5 random targets
  env.reset()
  ep_rewards = []
  done = False
  obs = env.reset()
  while not done:
      frame = env.sim.renderer.render_offscreen(camera_id=1,
                        width=400,
                        height=400)
      frames.append(frame)
      o = env.get_obs()
      # get the next action from the policy
      action, _ = model.predict(o)
      #print(pi.show_activations())
      #print(pi.get_neurons())
      # take an action based on the current observation
      obs, reward, done, info = env.step(action)

env.close()

c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:77: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")


In [14]:
import imageio

os.makedirs('videos', exist_ok=True)
video_path = 'videos/test.mp4'
# make a local copy
imageio.mimsave(video_path, frames, fps=30)
show_video('videos/test.mp4')